# LeetCode 519 — Random Flip Matrix bằng Python

Link đề: https://leetcode.com/problems/random-flip-matrix/ · Độ khó: **Medium** · Chủ đề: Randomized, Hash Table, Math, Reservoir Sampling (thiết kế lớp)

Skeleton LeetCode:

```python
class Solution:

    def __init__(self, m: int, n: int):
        ...

    def flip(self) -> List[int]:
        ...

    def reset(self) -> None:
        ...


# Your Solution object will be instantiated and called as such:
# obj = Solution(m, n)
# param_1 = obj.flip()
# obj.reset()
```

Ghi chú: bài **thiết kế lớp + ngẫu nhiên**. Đáp án của `flip()` không cố định, nên notebook test bằng **tính chất** (ô trả về hợp lệ, chưa lật, không trùng cho đến khi hết, `reset` hoạt động), **thống kê có seed** và **vét cạn mọi lựa chọn ngẫu nhiên** (bằng một bộ RNG tự dựng).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một lưới nhị phân `m x n` toàn `0`; `flip()` chọn **ngẫu nhiên đều** một ô đang là `0`, đổi nó thành `1` rồi trả toạ độ; `reset()` đưa cả lưới về `0`. Ngoài ra phải **ít gọi hàm random** và **ít bộ nhớ/thời gian** nhất có thể.
2. Nắm các công cụ Python: đổi toạ độ 2D ↔ chỉ số phẳng bằng `divmod`, `dict.get(key, mặc_định)`, `random.randint` (gồm cả hai đầu), `random.seed`, `bisect.insort`, và cách **tự dựng một bộ sinh số ngẫu nhiên có seed** (LCG) để test.
3. Đi từ **bốc bừa đến khi trúng ô trống** → **danh sách ô trống** → **tập ô đã lật + bốc lại** → **hạng thứ r trong các ô trống** → **Fisher–Yates "ảo" bằng dict**.
4. Hiểu mấu chốt: coi lưới là một đoạn `[0, m·n)` rồi **bốc đều từ phần còn lại**; chỉ ghi nhớ **những vị trí bị hoán đổi** thay vì cả mảng `m·n` phần tử (đến `10^8`).
5. Tránh các lỗi kinh điển: sai biên `randint`, quên cập nhật ánh xạ, `reset` không xoá ánh xạ, đổi chỉ số phẳng với `m` thay vì `n`, và test ngẫu nhiên bằng so sánh cứng.

## 2. Đề bài & ví dụ

Có một lưới nhị phân `m x n`, ban đầu **mọi ô đều `0`**. Cài lớp `Solution`:

- `Solution(m, n)`: khởi tạo với kích thước lưới.
- `flip()`: trả về một chỉ số `[i, j]` **ngẫu nhiên** sao cho `matrix[i][j] == 0`, rồi đổi ô đó thành `1`. **Mọi ô đang là `0` phải có xác suất như nhau.**
- `reset()`: đặt lại toàn bộ lưới về `0`.

Yêu cầu thêm: **tối thiểu số lần gọi hàm random có sẵn** của ngôn ngữ, và tối ưu thời gian/bộ nhớ.

**Ví dụ:**

```
Input
["Solution", "flip", "flip", "flip", "reset", "flip"]
[[3, 1], [], [], [], [], []]
Output
[null, [1, 0], [2, 0], [0, 0], null, [2, 0]]
```

Giải thích: lưới `3 x 1` có 3 ô `[0,0]`, `[1,0]`, `[2,0]`. Lần `flip` đầu bất kỳ ô nào cũng được (đề trả `[1,0]`); lần hai chỉ còn hai ô; lần ba chỉ còn đúng một ô (`[0,0]`). Sau `reset` mọi ô lại tự do, `flip` trả `[2,0]`. (Output trong đề chỉ là **một khả năng**: các ô trả về thay đổi theo từng lần chạy.)

**Ràng buộc:**

- `1 <= m, n <= 10^4` (tức tới `10^8` ô!).
- Mỗi lần gọi `flip` luôn còn ít nhất một ô `0`.
- Tối đa `1000` lần gọi `flip` và `reset`.

Cell dưới dựng "bộ chạy chuỗi lệnh" mô phỏng cách LeetCode gọi lớp, cùng hai bộ RNG tự dựng (giải thích ở mục 3.5) mà ta dùng suốt notebook.

In [1]:
import random
import time
import sys
from bisect import insort
from collections import Counter
from contextlib import contextmanager
from itertools import permutations, product
from math import factorial
from typing import List


def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp: ops[0] là hàm tạo."""
    obj, out = None, []
    for i, (op, a) in enumerate(zip(ops, args)):
        if i == 0:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out


class LCG:
    """RNG tự dựng, có seed: Linear Congruential Generator 64-bit (hằng số của Knuth/MMIX)."""

    MASK = (1 << 64) - 1

    def __init__(self, seed=1):
        self.state = seed & self.MASK
        self.calls = 0                      # đếm số lần gọi randint

    def _next53(self):
        self.state = (self.state * 6364136223846793005 + 1442695040888963407) & self.MASK
        return self.state >> 11             # lấy 53 bit cao (bit cao "đẹp" hơn bit thấp)

    def randint(self, a, b):
        """Số nguyên đều trong [a, b] (gồm cả b), giống random.randint."""
        self.calls += 1
        return a + int(self._next53() / (1 << 53) * (b - a + 1))


class ScriptedRNG:
    """RNG "kịch bản": trả đúng dãy giá trị ta định sẵn (để mô phỏng/vét cạn). Có kiểm tra biên."""

    def __init__(self, picks):
        self.picks, self.pos, self.calls = list(picks), 0, 0

    def randint(self, a, b):
        v = self.picks[self.pos]
        self.pos += 1
        self.calls += 1
        assert a <= v <= b, f"RNG được hỏi [{a},{b}] nhưng kịch bản đưa {v}"
        return v


ops = ["Solution", "flip", "flip", "flip", "reset", "flip"]
args = [[3, 1], [], [], [], [], []]
print(ops)
print(args)

['Solution', 'flip', 'flip', 'flip', 'reset', 'flip']
[[3, 1], [], [], [], [], []]


## 3. Kiến thức nền

### 3.1. Toạ độ 2D ↔ chỉ số phẳng (`divmod`)

Lưới `m x n` có `m·n` ô. Ta **đánh số các ô theo từng hàng**: ô `(i, j)` có số thứ tự `idx = i * n + j` (nhân với **số cột `n`**). Ngược lại `i = idx // n`, `j = idx % n`. Python có `divmod(a, b)` trả luôn cả thương và dư: `divmod(idx, n) = (idx // n, idx % n)`.

Ví dụ lưới `2 x 3` (m = 2 hàng, n = 3 cột): các ô được đánh số `0 1 2 / 3 4 5`.

In [2]:
m, n = 2, 3
for i in range(m):
    print([i * n + j for j in range(n)])
print(divmod(4, n))        # (1, 1): ô số 4 nằm ở hàng 1, cột 1
print(divmod(5, n))        # (1, 2)
print(1 * n + 2)           # 5: đổi ngược lại

[0, 1, 2]
[3, 4, 5]
(1, 1)
(1, 2)
5


Nhờ đó "chọn một ô ngẫu nhiên" trở thành "chọn một **số nguyên** trong `[0, m·n - 1]`" — một việc cần **đúng 1 lần gọi random** thay vì 2 lần (một cho hàng, một cho cột).

### 3.2. `random.randint` gồm cả hai đầu, và `random.seed`

`random.randint(a, b)` cho số nguyên đều trong **`[a, b]` — gồm cả `b`** (khác `range(a, b)` loại `b`). `random.seed(x)` cố định dãy số ngẫu nhiên để lần chạy sau cho kết quả giống hệt — rất tiện để debug.

In [3]:
random.seed(3)
print(sorted({random.randint(2, 5) for _ in range(300)}))      # [2, 3, 4, 5]
random.seed(42); a = [random.randint(0, 9) for _ in range(5)]
random.seed(42); b = [random.randint(0, 9) for _ in range(5)]
print(a, b, a == b)                                             # cùng seed -> cùng dãy

[2, 3, 4, 5]
[1, 0, 4, 3, 3] [1, 0, 4, 3, 3] True


### 3.3. `dict.get(key, mặc_định)` và "mảng ảo"

`d.get(k, mặc_định)` trả `d[k]` nếu có `k`, ngược lại trả `mặc_định` mà **không** thêm `k` vào `d`. Ta sẽ dùng `mp.get(x, x)` với nghĩa: "**nếu `x` chưa bị ghi đè thì giá trị tại vị trí `x` chính là `x`**". Nhờ vậy một dict **thưa** có thể đóng vai một mảng `0, 1, 2, ..., m·n-1` rất dài mà ta không bao giờ phải tạo ra.

In [4]:
mp = {}
print(mp.get(7, 7), len(mp))      # 7 0  (chưa ghi đè, mp vẫn rỗng)
mp[7] = 99
print(mp.get(7, 7), len(mp))      # 99 1

7 0
99 1


### 3.4. Ý tưởng Fisher–Yates "bốc rồi lấp chỗ trống"

Có một dãy `A = [0, 1, 2, 3, 4]`. Muốn bốc ngẫu nhiên một phần tử rồi **không bốc lại** nó: chọn vị trí `x` ngẫu nhiên trong phần còn lại, lấy `A[x]`, rồi **chép phần tử cuối cùng đè lên vị trí `x`** và coi như dãy ngắn đi 1. Không phải dịch chuyển cả dãy ⇒ `O(1)`.

In [5]:
A = [10, 20, 30, 40, 50]
size = len(A)
for x in (1, 0, 2):                    # giả sử ta bốc các vị trí 1, 0, 2
    size -= 1                          # phần còn lại là A[0..size]
    print("bốc", A[x], end=" | ")
    A[x] = A[size]                     # lấp chỗ trống bằng phần tử cuối của phần còn lại
    print("còn lại:", A[:size])

bốc 20 | còn lại: [10, 50, 30, 40]
bốc 10 | còn lại: [40, 50, 30]
bốc 30 | còn lại: [40, 50]


### 3.5. Tự dựng một RNG có seed (LCG) — để test thống kê và vét cạn

Hàm `random` của Python là hộp đen. Để **kiểm soát** tuyệt đối (đếm số lần gọi, tái lập, ép ra từng lựa chọn cụ thể) ta dựng hai bộ RNG nhỏ ở mục 2:

- `LCG(seed)`: **bộ đồng dư tuyến tính** `state ← (a·state + c) mod 2^64` với hằng số `a = 6364136223846793005`, `c = 1442695040888963407`; lấy 53 bit cao chia cho `2^53` được số thực trong `[0,1)`, nhân với độ rộng khoảng để ra `randint`. Có đếm `calls`.
- `ScriptedRNG(picks)`: **không ngẫu nhiên**, trả đúng dãy ta đưa — để mô phỏng và để **liệt kê mọi lựa chọn** (vét cạn).

Các lớp demo của ta nhận tham số `rng`, nên cắm được cả hai. Lớp nộp LeetCode sẽ dùng thẳng `random.randint`.

In [6]:
r = LCG(seed=2024)
print([r.randint(1, 6) for _ in range(12)], "calls =", r.calls)
r2 = LCG(seed=2024)
print([r2.randint(1, 6) for _ in range(12)])          # cùng seed -> giống hệt
# kiểm tra nhanh tính đều: 60000 lần tung xúc xắc
g = LCG(7)
cnt = Counter(g.randint(1, 6) for _ in range(60000))
print(sorted(cnt.items()))
s = ScriptedRNG([3, 0, 2])
print(s.randint(0, 5), s.randint(0, 5), s.randint(0, 5), "calls =", s.calls)

[3, 2, 5, 2, 6, 3, 1, 2, 6, 2, 2, 4] calls = 12
[3, 2, 5, 2, 6, 3, 1, 2, 6, 2, 2, 4]
[(1, 9913), (2, 10058), (3, 9886), (4, 9940), (5, 10138), (6, 10065)]
3 0 2 calls = 3


### 3.6. Kiểm tra "đều" bằng thống kê: quy tắc ≤ 5σ

Nếu có `k` kết quả đồng khả năng và ta lấy `T` mẫu, số lần xuất hiện của mỗi kết quả xấp xỉ `T/k` với độ lệch chuẩn `σ = sqrt(T · p · (1-p))`, `p = 1/k`. Một thuật toán đều thì **mọi** kết quả lệch không quá vài `σ` (ta chọn ngưỡng rộng tay `5σ` để test không "rớt oan"). Một thuật toán lệch (thiên vị) sẽ vượt ngưỡng khi `T` đủ lớn.

In [7]:
def worst_sigma(counts, keys, trials):
    """Độ lệch lớn nhất (đơn vị sigma) giữa số lần đếm được và kỳ vọng trials/k."""
    k = len(keys)
    p = 1 / k
    sigma = (trials * p * (1 - p)) ** 0.5 if k > 1 else 1
    assert set(counts) <= set(keys), "xuất hiện kết quả lạ ngoài tập hợp lệ"
    return max(abs(counts.get(key, 0) - trials * p) / sigma for key in keys)


g = LCG(11)
cnt = Counter(g.randint(0, 5) for _ in range(60000))
print("lệch tối đa:", round(worst_sigma(cnt, range(6), 60000), 2), "sigma")     # nhỏ (< 5)

lệch tối đa: 2.22 sigma


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng **lưới `2 x 3` = 6 ô giấy đánh số 0..5 đặt trong một cái hộp**. `flip()` = **nhắm mắt bốc một tờ giấy ra khỏi hộp** (ô đó "bị lật", không bỏ lại). `reset()` = bỏ tất cả giấy về hộp.

Tính tay xác suất:

- Lần `flip` thứ 1: hộp có 6 tờ ⇒ mỗi ô xác suất `1/6`.
- Giả sử bốc được ô `4`. Lần thứ 2: còn 5 tờ ⇒ mỗi ô còn lại `1/5`.
- Cứ thế, lần thứ 6 chỉ còn 1 tờ, xác suất `1`.

Xác suất cả chuỗi "4 rồi 1 rồi ..." là tích `1/6 · 1/5 · 1/4 · 1/3 · 1/2 · 1 = 1/720 = 1/6!`. Vậy `flip` đúng đắn chính là **xáo ngẫu nhiên đều dãy `0..m·n-1`** nhưng ta chỉ lấy ra từng phần tử một khi được hỏi.

Câu hỏi thực sự của bài: **làm sao mô phỏng "cái hộp" khi có tới `10^8` tờ giấy**, mà chỉ gọi random **đúng một lần** mỗi `flip` và chỉ lật tối đa `1000` ô?

## 5. Cách 1 — Brute Force / baseline: lưới thật + bốc bừa đến khi trúng ô `0`

**Ý tưởng:** tạo đúng ma trận `m x n` toàn `0`. `flip()`: lặp — bốc `i` ngẫu nhiên trong `[0, m-1]`, `j` ngẫu nhiên trong `[0, n-1]`; nếu `matrix[i][j] == 0` thì lật thành `1` và trả `[i, j]`; ngược lại **bốc lại**. `reset()`: tạo lại ma trận.

**Vì sao đều?** Mỗi lần thử, mọi ô trong cả lưới đều xác suất `1/(m·n)`. Ta chỉ **chấp nhận** khi trúng ô `0`; điều kiện hoá trên "trúng ô 0" cho phân phối **đều trên các ô `0`**.

Đây là cách ngây thơ dễ nghĩ nhất. Mỗi lần thử gọi random **2 lần**.

In [8]:
class MatrixDemo:
    """Cách 1 (demo): lưới thật + bốc bừa cho tới khi trúng ô 0."""

    def __init__(self, m, n, rng=random):
        self.m, self.n, self.rng = m, n, rng
        self.matrix = [[0] * n for _ in range(m)]
        self.tries = 0                         # đếm số lần thử (để quan sát nút thắt)

    def flip(self):
        while True:
            i = self.rng.randint(0, self.m - 1)
            j = self.rng.randint(0, self.n - 1)
            self.tries += 1
            if self.matrix[i][j] == 0:
                self.matrix[i][j] = 1
                return [i, j]

    def reset(self):
        self.matrix = [[0] * self.n for _ in range(self.m)]


g = LCG(5)
s = MatrixDemo(2, 3, g)
res = [s.flip() for _ in range(6)]
print(res, "| số lần thử:", s.tries, "| số lần gọi RNG:", g.calls)
assert sorted(map(tuple, res)) == [(i, j) for i in range(2) for j in range(3)]

[[1, 0], [0, 0], [0, 2], [1, 2], [0, 1], [1, 1]] | số lần thử: 23 | số lần gọi RNG: 46


**Độ phức tạp:** bộ nhớ **`O(m·n)`** (cả lưới); khởi tạo/`reset` `O(m·n)`; `flip` trung bình **không cố định**: càng nhiều ô đã lật, càng phải thử nhiều lần.

**Nút thắt (hai cái):**

1. **Bộ nhớ:** `m = n = 10^4` ⇒ `10^8` ô — tràn bộ nhớ ngay khi khởi tạo (mỗi phần tử list chiếm ≥ 8 byte ⇒ ≥ 800 MB).
2. **Số lần gọi random:** khi lưới gần đầy, tỉ lệ trúng ô `0` rất nhỏ. Khi còn `r` ô trống trong `N` ô, số lần thử kỳ vọng là `N/r` (phân phối hình học). Xem thí nghiệm ngay dưới.

In [9]:
def expected_attempts_all(N):
    """Kỳ vọng số lần thử để lật HẾT N ô bằng bốc bừa: N*(1 + 1/2 + ... + 1/N)  (bài 'sưu tầm tem')."""
    return N * sum(1 / k for k in range(1, N + 1))


N = 9
print("kỳ vọng số lần thử cho 3x3:", round(expected_attempts_all(N), 2), "(thay vì 9 nếu mỗi flip chỉ cần 1 lần)")

total = 0
for seed in range(2000):
    g = LCG(seed)
    s = MatrixDemo(3, 3, g)
    for _ in range(9):
        s.flip()
    total += s.tries
print("trung bình thực nghiệm:", round(total / 2000, 2), "lần thử; mỗi lần thử gọi random 2 lần")
print("lần flip cuối (còn 1 ô trống / 9) kỳ vọng thử:", 9 / 1, "lần")
print("RAM của một list 10^6 số nguyên:", sys.getsizeof(list(range(10**6))) // 10**6, "MB  ->  10^8 ô sẽ cần ~100 lần như thế")

kỳ vọng số lần thử cho 3x3: 25.46 (thay vì 9 nếu mỗi flip chỉ cần 1 lần)
trung bình thực nghiệm: 25.3 lần thử; mỗi lần thử gọi random 2 lần
lần flip cuối (còn 1 ô trống / 9) kỳ vọng thử: 9.0 lần


RAM của một list 10^6 số nguyên: 8 MB  ->  10^8 ô sẽ cần ~100 lần như thế


## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Phẳng hoá:** thay `(i, j)` bằng một số nguyên `idx ∈ [0, m·n)`. Bài trở thành **"bốc một số chưa bốc từ `0..N-1`, `N = m·n`"** và chỉ cần **một** lần random.
2. **Fisher–Yates ảo:** gọi `total` là số ô còn trống. Mỗi `flip`:
   - `total -= 1` (phần còn lại là `A[0..total]`),
   - bốc `x` đều trong `[0, total]`,
   - **kết quả là `A[x]`**,
   - rồi **lấp chỗ trống**: `A[x] = A[total]`.

   Mảng `A` ban đầu là "đồng nhất" `A[k] = k` nên ta **không cần tạo nó**: chỉ lưu những vị trí đã bị ghi đè trong dict `mp` (vị trí vắng mặt có `A[k] = k`). Sau `k` lần `flip`, `mp` có tối đa `k ≤ 1000` khoá!

Tóm tắt: **1 lần random mỗi `flip`**, **`O(1)` thời gian**, **bộ nhớ `O(số lần flip)`** — hoàn toàn không phụ thuộc `m·n`.

Phần dưới ta xem các cách trung gian (để thấy từng nút thắt được gỡ thế nào), rồi tới cách này.

## 7. Các cách tiếp theo

Tổng cộng 5 cách:

- Cách 1: lưới thật + bốc bừa (mục 5).
- Cách 2: danh sách ô trống + đổi với phần tử cuối (`O(m·n)` bộ nhớ nhưng đúng 1 lần random).
- Cách 3: tập ô đã lật + bốc lại (bộ nhớ nhỏ nhưng vẫn bốc lại).
- Cách 4: "hạng thứ r" trong các ô trống bằng danh sách đã lật được sắp xếp.
- Cách 5: Fisher–Yates ảo bằng dict (**chính thức**).

Mỗi cách có lớp **demo** (nhận `rng`, tiện đếm/ép lựa chọn) rồi lớp **nộp LeetCode** (đúng `class Solution`-style, dùng `random.randint`).

---

### Cách 2 — Danh sách ô trống + "đổi với phần tử cuối rồi `pop`"

Giải thích từng đoạn:

1. `__init__`: `free = list(range(m*n))` — danh sách mọi chỉ số ô còn trống.
2. `flip`: bốc `k` đều trong `[0, len(free)-1]` (**1 lần random**).
3. Đổi `free[k]` với `free[-1]`, rồi `pop()` phần tử cuối: `O(1)` (không dồn).
4. Đổi chỉ số phẳng ra `[i, j]` bằng `divmod`.
5. `reset`: dựng lại `free`.

In [10]:
class FreeListDemo:
    """Cách 2 (demo): list các ô trống; đổi chỗ với phần tử cuối rồi pop. Bộ nhớ O(m*n)."""

    def __init__(self, m, n, rng=random):
        self.m, self.n, self.rng = m, n, rng
        self.free = list(range(m * n))

    def flip(self):
        k = self.rng.randint(0, len(self.free) - 1)
        self.free[k], self.free[-1] = self.free[-1], self.free[k]
        idx = self.free.pop()
        return list(divmod(idx, self.n))

    def reset(self):
        self.free = list(range(self.m * self.n))


g = LCG(5)
s = FreeListDemo(2, 3, g)
res = [s.flip() for _ in range(6)]
print(res, "| số lần gọi RNG:", g.calls)
assert sorted(map(tuple, res)) == [(i, j) for i in range(2) for j in range(3)] and g.calls == 6

[[1, 1], [0, 0], [1, 0], [1, 2], [0, 2], [0, 1]] | số lần gọi RNG: 6


**Code nộp LeetCode cho Cách 2:**

```python
import random

class Solution:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.free = list(range(m * n))

    def flip(self) -> List[int]:
        k = random.randint(0, len(self.free) - 1)
        self.free[k], self.free[-1] = self.free[-1], self.free[k]
        idx = self.free.pop()
        return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.free = list(range(self.m * self.n))
```

**Độ phức tạp:** `flip` `O(1)`, **1 lần random**; nhưng khởi tạo/`reset` `O(m·n)` thời gian và bộ nhớ `O(m·n)` ⇒ `10^8` phần tử: **MLE/TLE** ở ràng buộc lớn nhất. Vẫn là bước đệm tốt: đây chính là Fisher–Yates thật sự mà Cách 5 sẽ "ảo hoá".

---

### Cách 3 — Tập ô đã lật + bốc lại

Giải thích từng đoạn:

1. Chỉ nhớ **các ô đã lật** trong một `set` `used` (nhớ chỉ số phẳng) — tối đa `1000` phần tử.
2. `flip`: bốc `idx` đều trong **toàn bộ** `[0, m·n-1]`; nếu `idx` đã trong `used` thì **bốc lại**.
3. Thêm `idx` vào `used`, trả `divmod`.
4. `reset`: `used.clear()`.

Giống Cách 1 về tư tưởng "bốc lại" nhưng **không giữ lưới** và chỉ **1 lần random mỗi lần thử**.

In [11]:
class SetRejectDemo:
    """Cách 3 (demo): set các ô đã lật; bốc cả khoảng rồi bốc lại nếu trúng ô đã lật."""

    def __init__(self, m, n, rng=random):
        self.m, self.n, self.rng = m, n, rng
        self.used = set()
        self.tries = 0

    def flip(self):
        while True:
            idx = self.rng.randint(0, self.m * self.n - 1)
            self.tries += 1
            if idx not in self.used:
                self.used.add(idx)
                return list(divmod(idx, self.n))

    def reset(self):
        self.used.clear()


g = LCG(5)
s = SetRejectDemo(2, 3, g)
res = [s.flip() for _ in range(6)]
print(res, "| số lần thử:", s.tries, "| số lần gọi RNG:", g.calls)
assert sorted(map(tuple, res)) == [(i, j) for i in range(2) for j in range(3)]

[[1, 1], [0, 0], [0, 1], [0, 2], [1, 2], [1, 0]] | số lần thử: 15 | số lần gọi RNG: 15


**Code nộp LeetCode cho Cách 3:**

```python
import random

class Solution:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.used = set()

    def flip(self) -> List[int]:
        while True:
            idx = random.randint(0, self.m * self.n - 1)
            if idx not in self.used:
                self.used.add(idx)
                return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.used.clear()
```

**Độ phức tạp:** bộ nhớ `O(k)` (`k` = số ô đã lật), `reset` `O(k)`. Mỗi lần thử `O(1)`. Nếu `k ≪ m·n` (như `k ≤ 1000` với lưới lớn) thì **gần như không bao giờ phải bốc lại**. Nhưng với lưới nhỏ (ví dụ `1 x 1000` bị lật gần hết) số lần random **không có cận trên cố định** (kỳ vọng `N/(N-k)` mỗi lần) — vi phạm yêu cầu "tối thiểu số lần gọi random".

---

### Cách 4 — "Hạng thứ r" trong các ô trống + danh sách đã lật được sắp xếp

Đây là cách **khác về ý tưởng**: không bốc lại, không đổi chỗ, mà bốc **hạng** rồi **dịch hạng thành chỉ số thật**.

Giải thích từng đoạn:

1. Giữ danh sách `used` **đã sắp xếp tăng** các chỉ số ô đã lật.
2. Có `free = m·n - len(used)` ô trống. Bốc `r` đều trong `[0, free-1]`: ta muốn ô trống **thứ `r`** (đếm từ 0).
3. Dịch: bắt đầu `idx = r`; duyệt `u` trong `used` theo thứ tự tăng: nếu `u <= idx` thì có một ô đã lật nằm trước/trúng vị trí này, ô trống thứ `r` phải **đẩy lên một**: `idx += 1`; ngược lại dừng. (Vì `used` tăng dần nên đúng.)
4. `insort(used, idx)` để giữ danh sách có thứ tự.

Ví dụ: lưới 6 ô, `used = [1, 4]`; ô trống là `0, 2, 3, 5`. Với `r = 2` ta muốn `3`: `idx = 2`; `u = 1 <= 2` ⇒ `idx = 3`; `u = 4 > 3` dừng ⇒ `3`. ✓

In [12]:
class RankDemo:
    """Cách 4 (demo): bốc hạng r trong các ô trống, dịch qua danh sách đã lật (sắp xếp)."""

    def __init__(self, m, n, rng=random):
        self.m, self.n, self.rng = m, n, rng
        self.used = []                                   # luôn sắp xếp tăng

    def flip(self):
        free = self.m * self.n - len(self.used)
        r = self.rng.randint(0, free - 1)                # hạng r trong các ô trống
        idx = r
        for u in self.used:
            if u <= idx:
                idx += 1
            else:
                break
        insort(self.used, idx)
        return list(divmod(idx, self.n))

    def reset(self):
        self.used = []


# kiểm tra đúng ví dụ ở trên: used=[1,4], r=2 -> ô 3
s = RankDemo(2, 3, ScriptedRNG([2]))
s.used = [1, 4]
print(s.flip(), s.used)       # ô 3 -> [1, 0]; used = [1, 3, 4]
g = LCG(5)
s = RankDemo(2, 3, g)
res = [s.flip() for _ in range(6)]
print(res, "| số lần gọi RNG:", g.calls)
assert sorted(map(tuple, res)) == [(i, j) for i in range(2) for j in range(3)] and g.calls == 6

[1, 0] [1, 3, 4]
[[1, 1], [0, 0], [1, 2], [0, 1], [0, 2], [1, 0]] | số lần gọi RNG: 6


**Code nộp LeetCode cho Cách 4:**

```python
import random
from bisect import insort

class Solution:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.used = []

    def flip(self) -> List[int]:
        idx = random.randint(0, self.m * self.n - len(self.used) - 1)
        for u in self.used:
            if u <= idx:
                idx += 1
            else:
                break
        insort(self.used, idx)
        return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.used = []
```

**Độ phức tạp:** **1 lần random mỗi `flip`**, bộ nhớ `O(k)`, nhưng thời gian `flip` là `O(k)` (duyệt + `insort`) ⇒ tổng `O(k²)` — với `k ≤ 1000` vẫn rất nhanh, nhưng chưa phải `O(1)`.

---

### Cách 5 — Fisher–Yates ảo bằng dict (chính thức)

Giải thích từng đoạn (khớp mục 6):

1. `total = m·n` = số ô còn trống; `mp = {}` = "những vị trí đã bị ghi đè".
2. `flip`: `total -= 1`; `x = random.randint(0, total)` (gồm cả `total`).
3. `idx = mp.get(x, x)`: giá trị đang nằm ở vị trí `x` — nếu `x` chưa từng bị ghi đè thì là chính `x`.
4. Lấp chỗ trống: `mp[x] = mp.get(total, total)` — chép giá trị của phần tử **cuối** (vị trí `total`) đè lên vị trí `x`.
5. Trả `[idx // n, idx % n]`.
6. `reset`: `total = m·n`, `mp.clear()`.

Ta không cần xoá khoá `total` khỏi `mp`: vị trí `≥ total` sẽ không bao giờ được truy vấn lại (vì `total` chỉ giảm tới khi `reset`).

In [13]:
class VirtualFYDemo:
    """Cách 5 (demo): Fisher-Yates ảo. Mảng A[k]=k ngầm, chỉ lưu các vị trí bị ghi đè trong mp."""

    def __init__(self, m, n, rng=random):
        self.m, self.n, self.rng = m, n, rng
        self.total = m * n
        self.mp = {}

    def flip(self):
        self.total -= 1
        x = self.rng.randint(0, self.total)
        idx = self.mp.get(x, x)
        self.mp[x] = self.mp.get(self.total, self.total)
        return [idx // self.n, idx % self.n]

    def reset(self):
        self.total = self.m * self.n
        self.mp.clear()


g = LCG(5)
s = VirtualFYDemo(2, 3, g)
res = [s.flip() for _ in range(6)]
print(res, "| số lần gọi RNG:", g.calls, "| mp có", len(s.mp), "khoá")
assert sorted(map(tuple, res)) == [(i, j) for i in range(2) for j in range(3)] and g.calls == 6

[[1, 1], [0, 0], [1, 0], [1, 2], [0, 2], [0, 1]] | số lần gọi RNG: 6 | mp có 3 khoá


**Độ phức tạp:** `flip` `O(1)` thời gian (hai/ba lần tra dict), **đúng 1 lần random**, bộ nhớ `O(k)` với `k` = số lần `flip` kể từ lần `reset` gần nhất (tối đa 1000). `reset` `O(k)` (clear dict). Đây là cách tối ưu ở mọi trục và là **code nộp** ở mục 9.

In [14]:
# Các lớp "nộp LeetCode" cho 4 cách đầu (dùng random.randint toàn cục) — để mục 12 test chung.
class SolMatrix:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.matrix = [[0] * n for _ in range(m)]

    def flip(self) -> List[int]:
        while True:
            i = random.randint(0, self.m - 1)
            j = random.randint(0, self.n - 1)
            if self.matrix[i][j] == 0:
                self.matrix[i][j] = 1
                return [i, j]

    def reset(self) -> None:
        self.matrix = [[0] * self.n for _ in range(self.m)]


class SolFreeList:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.free = list(range(m * n))

    def flip(self) -> List[int]:
        k = random.randint(0, len(self.free) - 1)
        self.free[k], self.free[-1] = self.free[-1], self.free[k]
        idx = self.free.pop()
        return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.free = list(range(self.m * self.n))


class SolSetReject:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.used = set()

    def flip(self) -> List[int]:
        while True:
            idx = random.randint(0, self.m * self.n - 1)
            if idx not in self.used:
                self.used.add(idx)
                return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.used.clear()


class SolRank:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.used = []

    def flip(self) -> List[int]:
        idx = random.randint(0, self.m * self.n - len(self.used) - 1)
        for u in self.used:
            if u <= idx:
                idx += 1
            else:
                break
        insort(self.used, idx)
        return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.used = []


print("đã định nghĩa 4 lớp nộp: SolMatrix, SolFreeList, SolSetReject, SolRank")

đã định nghĩa 4 lớp nộp: SolMatrix, SolFreeList, SolSetReject, SolRank


## 8. TRACE từng bước (Cách 5)

Lưới `m = 2`, `n = 3` (6 ô đánh số `0 1 2 / 3 4 5`). Ta **ép** dãy lựa chọn `x = 4, 4, 1, 1, 0, 0` bằng `ScriptedRNG` để xem rõ các nhánh: chọn vị trí bị ghi đè, chọn đúng vị trí cuối (`x == total`), v.v.

Quy ước: `A` là mảng ảo `A[k] = mp.get(k, k)`; phần còn trống là `A[0..total]`.

| lần | total (sau `-=1`) | x | `idx = mp.get(x, x)` | `mp[x] ← mp.get(total, total)` | mp sau | trả về |
|---|---|---|---|---|---|---|
| 1 | 5 | 4 | 4 | 5 | `{4:5}` | `[1,1]` |
| 2 | 4 | 4 | `mp[4]=5` | `mp.get(4)=5` | `{4:5}` | `[1,2]` |
| 3 | 3 | 1 | 1 | `3` | `{4:5, 1:3}` | `[0,1]` |
| 4 | 2 | 1 | `mp[1]=3` | `2` | `{4:5, 1:2}` | `[1,0]` |
| 5 | 1 | 0 | 0 | `mp.get(1)=2` | `{4:5, 1:2, 0:2}` | `[0,0]` |
| 6 | 0 | 0 | `mp[0]=2` | `mp.get(0)=2` | `{4:5, 1:2, 0:2}` | `[0,2]` |

Nhận xét: sau 6 lần ta nhận đúng 6 ô khác nhau `{[1,1],[1,2],[0,1],[1,0],[0,0],[0,2]}`. Lần 2 là trường hợp `x == total` (bốc đúng phần tử cuối: không cần "lấp chỗ" thật sự). Hàm `trace_flip` bên dưới in từng bước để bạn đối chiếu bảng.

In [15]:
def trace_flip(m, n, picks):
    """In từng bước Cách 5 với dãy lựa chọn ép sẵn (ScriptedRNG)."""
    rng = ScriptedRNG(picks)
    total, mp = m * n, {}
    print(f"lưới {m}x{n}, total={total}, mp={mp}")
    result = []
    for step in range(1, len(picks) + 1):
        total -= 1
        x = rng.randint(0, total)
        idx = mp.get(x, x)
        last = mp.get(total, total)
        mp[x] = last
        cell = [idx // n, idx % n]
        result.append(cell)
        print(f"lần {step}: total={total} x={x} idx=mp.get({x},{x})={idx} "
              f"mp[{x}]<-mp.get({total},{total})={last}  mp={mp}  -> {cell}")
    return result


res = trace_flip(2, 3, [4, 4, 1, 1, 0, 0])
assert len({tuple(c) for c in res}) == 6

lưới 2x3, total=6, mp={}
lần 1: total=5 x=4 idx=mp.get(4,4)=4 mp[4]<-mp.get(5,5)=5  mp={4: 5}  -> [1, 1]
lần 2: total=4 x=4 idx=mp.get(4,4)=5 mp[4]<-mp.get(4,4)=5  mp={4: 5}  -> [1, 2]
lần 3: total=3 x=1 idx=mp.get(1,1)=1 mp[1]<-mp.get(3,3)=3  mp={4: 5, 1: 3}  -> [0, 1]
lần 4: total=2 x=1 idx=mp.get(1,1)=3 mp[1]<-mp.get(2,2)=2  mp={4: 5, 1: 2}  -> [1, 0]
lần 5: total=1 x=0 idx=mp.get(0,0)=0 mp[0]<-mp.get(1,1)=2  mp={4: 5, 1: 2, 0: 2}  -> [0, 0]
lần 6: total=0 x=0 idx=mp.get(0,0)=2 mp[0]<-mp.get(0,0)=2  mp={4: 5, 1: 2, 0: 2}  -> [0, 2]


Trace thứ hai: **ví dụ đề** (`3 x 1`) với kịch bản `[1, 1, 0]`, rồi `reset`, rồi `flip` với `x = 2` — tái tạo **đúng** output mẫu `[[1,0],[2,0],[0,0], reset, [2,0]]`.

In [16]:
sol = VirtualFYDemo(3, 1, ScriptedRNG([1, 1, 0, 2]))
out = [sol.flip(), sol.flip(), sol.flip()]
print(out, "mp:", sol.mp)
sol.reset()
print("sau reset: total =", sol.total, "mp =", sol.mp)
out.append(sol.flip())
print(out)
assert out == [[1, 0], [2, 0], [0, 0], [2, 0]]

[[1, 0], [2, 0], [0, 0]] mp: {1: 2, 0: 0}
sau reset: total = 3 mp = {}
[[1, 0], [2, 0], [0, 0], [2, 0]]


## 9. Code nộp LeetCode

Đúng `class Solution`, phương thức `flip` và `reset`. Thêm `import random` ở đầu phòng khi môi trường chưa nạp sẵn.

In [17]:
class Solution:
    def __init__(self, m: int, n: int):
        self.m = m
        self.n = n
        self.total = m * n
        self.mp = {}

    def flip(self) -> List[int]:
        self.total -= 1
        x = random.randint(0, self.total)
        idx = self.mp.get(x, x)
        self.mp[x] = self.mp.get(self.total, self.total)
        return [idx // self.n, idx % self.n]

    def reset(self) -> None:
        self.total = self.m * self.n
        self.mp.clear()


@contextmanager
def scripted_random(picks):
    """Tạm thay random.randint bằng kịch bản để chạy lại ví dụ của đề trên lớp Solution nộp thật."""
    old = random.randint
    rng = ScriptedRNG(picks)
    random.randint = rng.randint
    try:
        yield rng
    finally:
        random.randint = old


with scripted_random([1, 1, 0, 2]):
    print(run_ops(Solution, ops, args))          # [None, [1,0], [2,0], [0,0], None, [2,0]]

random.seed(1)
print(run_ops(Solution, ops, args))              # chạy với random thật (kết quả là một khả năng bất kỳ)

[None, [1, 0], [2, 0], [0, 0], None, [2, 0]]
[None, [0, 0], [2, 0], [1, 0], None, [0, 0]]


## 10. Vì sao đúng

Gọi `A` là mảng ảo `A[k] = mp.get(k, k)`. Bất biến khi bắt đầu mỗi `flip`: **`A[0..total-1]` là một hoán vị của đúng các ô chưa lật** (mỗi ô trống xuất hiện đúng một lần).

- **Khởi tạo:** `total = m·n`, `mp = {}` ⇒ `A[k] = k` với `k = 0..m·n-1` — đủ mọi ô, mỗi ô một lần. ✓
- **Bước `flip`:** `total` giảm 1; bốc `x` đều trong `[0, total]` — tức đều trong `total + 1` ô còn trống (vị trí `0..total` trước khi thu nhỏ). Ta trả `A[x]`: **mỗi ô trống xác suất `1/(total+1)`**, tức đều trên các ô còn trống. ✓
- **Giữ bất biến:** gán `A[x] = A[total]` (phần tử cuối) rồi **bỏ vị trí `total`** ra khỏi vùng hợp lệ: ô vừa lật `A[x]` biến mất, phần tử cuối `A[total]` đến lấp chỗ — phần còn lại `A[0..total-1]` vẫn là hoán vị của các ô chưa lật. (Khi `x == total` ta gán `A[x] = A[x]`, không đổi gì, vẫn đúng.) ✓
- **Không trùng:** vì `A[0..total-1]` chỉ chứa các ô chưa lật, một ô đã trả về không bao giờ xuất hiện lại cho tới `reset`. Sau `m·n` lần `flip` mọi ô đã bị lật đúng một lần.
- **Đều trên cả chuỗi:** các lựa chọn `(x_1, x_2, …, x_N)` có `N · (N-1) ⋯ 1 = N!` tổ hợp đồng khả năng; mỗi tổ hợp cho **một thứ tự lật khác nhau** ⇒ mọi thứ tự lật xác suất `1/N!`. Ta sẽ kiểm chứng bằng vét cạn ở mục 12.
- **Đúng 1 lần random mỗi `flip`:** thấy ngay từ code; ta cũng sẽ đếm bằng `LCG.calls`.
- **`reset`:** đặt `total` về `m·n` và xoá `mp` ⇒ trở lại đúng trạng thái ban đầu.

## 11. Phân tích Time / Space Complexity

Đặt `N = m·n`, `k` = số lần `flip` kể từ `reset` gần nhất (`k ≤ 1000`).

| Cách | `flip` Time | `reset` Time | Space | Số lần random / `flip` |
|---|---|---|---|---|
| 1. Lưới + bốc bừa | kỳ vọng `N/(N-k)` lần thử | `O(N)` | `O(N)` | 2 × số lần thử |
| 2. List ô trống | `O(1)` | `O(N)` | `O(N)` | **1** |
| 3. Set + bốc lại | kỳ vọng `N/(N-k)` lần thử, mỗi lần `O(1)` | `O(k)` | `O(k)` | 1 × số lần thử |
| 4. Hạng thứ r | `O(k)` | `O(1)` | `O(k)` | **1** |
| 5. Fisher–Yates ảo (dict) | `O(1)` | `O(k)` | `O(k)` | **1** |

Cách 5 là duy nhất đạt **đồng thời**: `O(1)` mỗi `flip`, bộ nhớ chỉ theo số lần lật (không theo `m·n`), và đúng một lần random. Với `m = n = 10^4` ta không thể tạo bất cứ cấu trúc nào cỡ `N = 10^8`.

## 12. Bộ test

Chiến lược (bài ngẫu nhiên: **tính chất + thống kê có seed + vét cạn**, không so đáp án cứng):

1. **Ví dụ đề:** ép lựa chọn bằng `ScriptedRNG` để lặp lại đúng output mẫu (đã làm ở mục 9); thêm kiểm tra tính chất cho mọi cách.
2. **Tính chất:** mọi ô trả về nằm trong lưới, chưa lật, **không trùng** cho tới khi hết; sau `m·n` lần thì **chính xác phủ toàn lưới**; `reset` cho phép lật lại từ đầu.
3. **Edge case:** `1 x 1`, `1 x N`, `N x 1`, lưới vuông, lưới chữ nhật không vuông (bắt lỗi nhầm `m`/`n`).
4. **Thống kê có seed:** lần `flip` đầu đều trên mọi ô; lần `flip` thứ hai đều trên các ô còn lại; **toàn bộ thứ tự lật** của lưới `2 x 2` (24 khả năng) đều.
5. **Vét cạn:** dùng `ScriptedRNG` liệt kê **mọi** dãy lựa chọn cho Cách 2, 4, 5: mỗi thứ tự lật xuất hiện **đúng một lần**.
6. **Đếm lời gọi random** bằng `LCG.calls`; **ngẫu nhiên có seed đúng ràng buộc**; thời gian với `m = n = 10^4`.

In [18]:
ALL = [SolMatrix, SolFreeList, SolSetReject, SolRank, Solution]
DEMOS = [MatrixDemo, FreeListDemo, SetRejectDemo, RankDemo, VirtualFYDemo]


def check_full_cycle(cls, m, n, rounds=2):
    """Lật hết m*n ô, rồi reset, lặp `rounds` lần: ô hợp lệ, không trùng, phủ kín lưới."""
    s = cls(m, n)
    for _ in range(rounds):
        seen = set()
        for _ in range(m * n):
            i, j = s.flip()
            assert isinstance(i, int) and isinstance(j, int)
            assert 0 <= i < m and 0 <= j < n, (i, j)
            assert (i, j) not in seen, "trả lại ô đã lật!"
            seen.add((i, j))
        assert seen == {(i, j) for i in range(m) for j in range(n)}
        s.reset()


# 1) + 3) tính chất & edge case, mọi cách, nhiều hình dạng lưới
random.seed(519)
shapes = [(1, 1), (1, 2), (2, 1), (1, 7), (7, 1), (2, 2), (3, 3), (2, 5), (5, 2), (4, 6), (6, 4), (3, 7)]
for cls in ALL:
    for m, n in shapes:
        check_full_cycle(cls, m, n)
print("tính chất + edge case:", len(ALL), "cách x", len(shapes), "hình dạng: OK")

# reset giữa chừng
for cls in ALL:
    s = cls(3, 3)
    first = {tuple(s.flip()) for _ in range(5)}
    s.reset()
    again = {tuple(s.flip()) for _ in range(9)}
    assert len(first) == 5 and len(again) == 9
print("reset giữa chừng: OK")

tính chất + edge case: 5 cách x 12 hình dạng: OK
reset giữa chừng: OK


In [19]:
# 4a) lần flip đầu đều trên lưới 3x4 (12 ô), mọi cách
m, n, trials = 3, 4, 60000
cells = [(i, j) for i in range(m) for j in range(n)]
for cls in ALL:
    random.seed(1001)
    cnt = Counter()
    for _ in range(trials):
        cnt[tuple(cls(m, n).flip())] += 1
    w = worst_sigma(cnt, cells, trials)
    assert w < 5, (cls.__name__, w)
    print(f"{cls.__name__:<13} lần flip đầu đều trên {len(cells)} ô, lệch tối đa {w:.2f} sigma")

SolMatrix     lần flip đầu đều trên 12 ô, lệch tối đa 2.39 sigma


SolFreeList   lần flip đầu đều trên 12 ô, lệch tối đa 1.34 sigma


SolSetReject  lần flip đầu đều trên 12 ô, lệch tối đa 1.34 sigma
SolRank       lần flip đầu đều trên 12 ô, lệch tối đa 1.34 sigma


Solution      lần flip đầu đều trên 12 ô, lệch tối đa 1.34 sigma


In [20]:
# 4b) lần flip THỨ HAI: sau khi ô a bị lật, 11 ô còn lại phải đều (điều kiện hoá theo ô đầu)
m, n, trials = 3, 4, 66000
for cls in ALL:
    random.seed(2002)
    cnt, base = Counter(), 0
    for _ in range(trials):
        s = cls(m, n)
        a = tuple(s.flip())
        b = tuple(s.flip())
        assert a != b
        if a == (0, 0):                       # chỉ thống kê khi ô đầu tiên là (0,0)
            cnt[b] += 1
            base += 1
    keys = [c for c in cells if c != (0, 0)]
    w = worst_sigma(cnt, keys, base)
    assert w < 5, (cls.__name__, w)
    print(f"{cls.__name__:<13} sau ô (0,0): {base} mẫu, 11 ô còn lại đều (lệch {w:.2f} sigma)")

SolMatrix     sau ô (0,0): 5534 mẫu, 11 ô còn lại đều (lệch 1.27 sigma)


SolFreeList   sau ô (0,0): 5475 mẫu, 11 ô còn lại đều (lệch 2.88 sigma)
SolSetReject  sau ô (0,0): 5475 mẫu, 11 ô còn lại đều (lệch 2.60 sigma)


SolRank       sau ô (0,0): 5475 mẫu, 11 ô còn lại đều (lệch 2.88 sigma)
Solution      sau ô (0,0): 5475 mẫu, 11 ô còn lại đều (lệch 2.88 sigma)


In [21]:
# 4c) cả thứ tự lật của lưới 2x2: 4! = 24 thứ tự phải đều nhau
m, n, trials = 2, 2, 72000
orders = list(permutations([(0, 0), (0, 1), (1, 0), (1, 1)]))
for cls in ALL:
    random.seed(3003)
    cnt = Counter()
    for _ in range(trials):
        s = cls(m, n)
        cnt[tuple(tuple(s.flip()) for _ in range(4))] += 1
    w = worst_sigma(cnt, orders, trials)
    assert w < 5, (cls.__name__, w)
    print(f"{cls.__name__:<13} 24 thứ tự lật đều, lệch tối đa {w:.2f} sigma")

SolMatrix     24 thứ tự lật đều, lệch tối đa 2.37 sigma


SolFreeList   24 thứ tự lật đều, lệch tối đa 2.26 sigma


SolSetReject  24 thứ tự lật đều, lệch tối đa 2.29 sigma


SolRank       24 thứ tự lật đều, lệch tối đa 2.26 sigma


Solution      24 thứ tự lật đều, lệch tối đa 2.26 sigma


In [22]:
# 5) VÉT CẠN: mọi dãy lựa chọn của RNG -> mỗi thứ tự lật xuất hiện đúng 1 lần (Cách 2, 4, 5)
def all_orders(demo_cls, m, n):
    N = m * n
    ranges = [range(N - t) for t in range(N)]            # lần flip thứ t có (N - t) lựa chọn
    cnt = Counter()
    for picks in product(*ranges):
        s = demo_cls(m, n, ScriptedRNG(picks))           # ScriptedRNG tự kiểm tra biên của mỗi lần bốc
        cnt[tuple(tuple(s.flip()) for _ in range(N))] += 1
    return cnt


for demo_cls in (FreeListDemo, RankDemo, VirtualFYDemo):
    for m, n in [(2, 2), (1, 5), (2, 3)]:
        cnt = all_orders(demo_cls, m, n)
        N = m * n
        assert len(cnt) == factorial(N) and set(cnt.values()) == {1}, (demo_cls.__name__, m, n)
    print(f"{demo_cls.__name__:<15} vét cạn (2x2: 24, 1x5: 120, 2x3: 720 dãy lựa chọn): mỗi thứ tự lật đúng 1 lần")

# trên lớp Solution NỘP THẬT (dùng random.randint): ép bằng scripted_random
m, n = 2, 2
N = m * n
cnt = Counter()
for picks in product(*[range(N - t) for t in range(N)]):
    with scripted_random(picks):
        s = Solution(m, n)
        cnt[tuple(tuple(s.flip()) for _ in range(N))] += 1
assert len(cnt) == 24 and set(cnt.values()) == {1}
print("Solution (nộp thật) vét cạn 2x2: OK")

FreeListDemo    vét cạn (2x2: 24, 1x5: 120, 2x3: 720 dãy lựa chọn): mỗi thứ tự lật đúng 1 lần
RankDemo        vét cạn (2x2: 24, 1x5: 120, 2x3: 720 dãy lựa chọn): mỗi thứ tự lật đúng 1 lần
VirtualFYDemo   vét cạn (2x2: 24, 1x5: 120, 2x3: 720 dãy lựa chọn): mỗi thứ tự lật đúng 1 lần
Solution (nộp thật) vét cạn 2x2: OK


In [23]:
# 6a) đếm số lần gọi random: Cách 2, 4, 5 đúng 1 lần/flip; Cách 1, 3 nhiều hơn
N_cells = 9
rows = []
for demo_cls in DEMOS:
    total_calls = 0
    for seed in range(500):
        g = LCG(seed)
        s = demo_cls(3, 3, g)
        for _ in range(N_cells):
            s.flip()
        total_calls += g.calls
    rows.append((demo_cls.__name__, total_calls / 500))
for name, avg in rows:
    print(f"{name:<15} trung bình {avg:5.1f} lần gọi RNG để lật hết 9 ô (tối thiểu = 9)")
assert [round(a, 6) for _, a in rows if _ in ("FreeListDemo", "RankDemo", "VirtualFYDemo")] == [9.0, 9.0, 9.0]

# 6b) ngẫu nhiên có seed, đúng ràng buộc: m, n trong [1, 10^4], số flip <= min(m*n, 1000)
rng = random.Random(777)
for _ in range(200):
    m, n = rng.randint(1, 10 ** 4), rng.randint(1, 10 ** 4)
    k = min(m * n, rng.randint(1, 1000))
    s = Solution(m, n)
    seen = set()
    for _ in range(k):
        i, j = s.flip()
        assert 0 <= i < m and 0 <= j < n and (i, j) not in seen
        seen.add((i, j))
    assert len(s.mp) <= k
    if rng.random() < 0.3:
        s.reset()
        assert s.total == m * n and not s.mp
print("200 lưới ngẫu nhiên (m, n tới 10^4): OK")

t = time.perf_counter()
s = Solution(10 ** 4, 10 ** 4)
for _ in range(1000):
    s.flip()
s.reset()
for _ in range(1000):
    s.flip()
print(f"lưới 10^4 x 10^4, 2000 lần flip + reset: {time.perf_counter() - t:.4f}s, mp có {len(s.mp)} khoá")

MatrixDemo      trung bình  50.1 lần gọi RNG để lật hết 9 ô (tối thiểu = 9)
FreeListDemo    trung bình   9.0 lần gọi RNG để lật hết 9 ô (tối thiểu = 9)
SetRejectDemo   trung bình  26.1 lần gọi RNG để lật hết 9 ô (tối thiểu = 9)
RankDemo        trung bình   9.0 lần gọi RNG để lật hết 9 ô (tối thiểu = 9)
VirtualFYDemo   trung bình   9.0 lần gọi RNG để lật hết 9 ô (tối thiểu = 9)


200 lưới ngẫu nhiên (m, n tới 10^4): OK
lưới 10^4 x 10^4, 2000 lần flip + reset: 0.0015s, mp có 1000 khoá


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sai biên `randint` — bốc `x` trong `[0, total]` **trước** khi giảm `total`

Viết `x = random.randint(0, self.total)` rồi mới `self.total -= 1` (hoặc dùng `randrange(0, total)` rồi cũng trừ như `randint`): `randint` **gồm cả hai đầu** nên ta có thể bốc `x = total` — vị trí **đã nằm ngoài** phần còn lại. Hậu quả: trả về một ô đã lật (**trùng**) hoặc ô không tồn tại. Ép `x = total` để thấy:

In [24]:
class WrongBoundary:
    def __init__(self, m, n, rng):
        self.m, self.n, self.rng = m, n, rng
        self.total = m * n
        self.mp = {}

    def flip(self):                                    # SAI: bốc trước khi giảm total
        x = self.rng.randint(0, self.total)            # có thể trúng x = total (ngoài vùng hợp lệ)
        self.total -= 1
        idx = self.mp.get(x, x)
        self.mp[x] = self.mp.get(self.total, self.total)
        return [idx // self.n, idx % self.n]


s = WrongBoundary(1, 3, ScriptedRNG([3]))     # lưới 1x3 chỉ có ô 0,1,2 nhưng x = 3 được phép
print(s.flip())                               # [1, 0]: hàng 1 không tồn tại trong lưới 1 hàng!

[1, 0]


**Vì sao sai:** vùng hợp lệ trước khi bốc là `A[0..total-1]`, tức `total` phần tử; `randint(0, total)` cho `total + 1` giá trị. **Sửa:** giảm `total` **trước** rồi `randint(0, total)` (như code nộp), hoặc `randint(0, total - 1)` rồi mới giảm — nhưng chú ý khi đó vị trí "cuối" là `total - 1`.

### Lỗi 2: quên "lấp chỗ trống" (không cập nhật `mp[x]`) ⇒ trả lại ô cũ

Nếu chỉ `idx = mp.get(x, x)` mà **không** ghi `mp[x] = ...`, ô `idx` vẫn còn nằm ở vị trí `x` và có thể được bốc lại ở lần sau — **trùng**.

In [25]:
class ForgotUpdate:
    def __init__(self, m, n, rng):
        self.m, self.n, self.rng = m, n, rng
        self.total = m * n
        self.mp = {}

    def flip(self):
        self.total -= 1
        x = self.rng.randint(0, self.total)
        idx = self.mp.get(x, x)                        # SAI: thiếu dòng self.mp[x] = ...
        return [idx // self.n, idx % self.n]


s = ForgotUpdate(1, 3, ScriptedRNG([0, 0]))            # bốc vị trí 0 hai lần liên tiếp
print(s.flip(), s.flip())                              # [0,0] rồi lại [0,0]  -> TRÙNG

[0, 0] [0, 0]


**Vì sao sai:** sau khi lấy `A[x]` ta phải **loại nó khỏi vùng còn lại** bằng cách chép phần tử cuối đè lên `x` (`mp[x] = A[total]`). **Sửa:** thêm `self.mp[x] = self.mp.get(self.total, self.total)`.

### Lỗi 3: `reset` không xoá `mp` (hoặc không đặt lại `total`)

`reset` chỉ `self.total = m*n` mà quên `mp.clear()`: các ánh xạ cũ làm mảng ảo **không còn là hoán vị** của `0..N-1` ⇒ có ô không bao giờ xuất hiện, có ô xuất hiện hai lần trong cùng một chu kỳ.

In [26]:
class BadReset(Solution):
    def reset(self):
        self.total = self.m * self.n                   # SAI: quên self.mp.clear()


def count_bad(cls, trials=2000):
    """Đếm số lần một chu kỳ lật-hết (sau 1 lần reset) bị trùng ô."""
    bad = 0
    for _ in range(trials):
        s = cls(2, 2)
        for _ in range(4):
            s.flip()
        s.reset()
        got = [tuple(s.flip()) for _ in range(4)]
        if len(set(got)) != 4:
            bad += 1
    return bad


random.seed(5)
print("BadReset: số chu kỳ bị trùng ô:", count_bad(BadReset), "/ 2000")
print("Solution :", count_bad(Solution), "/ 2000")

BadReset: số chu kỳ bị trùng ô: 1917 / 2000


Solution : 0 / 2000


**Sửa:** `reset` phải đưa **mọi** trạng thái về như `__init__`: `total = m*n` **và** `mp.clear()`.

### Lỗi 4: đổi chỉ số phẳng bằng `m` thay vì `n` (chỉ lộ ra khi lưới **không vuông**)

`idx = i * n + j` nên phải `i = idx // n`, `j = idx % n` (chia cho **số cột**). Viết `idx // m` cho kết quả đúng ở lưới vuông (nên test mẫu `3 x 3` vẫn qua!) nhưng sai ở lưới chữ nhật: hàng/cột vượt biên hoặc trùng ô.

In [27]:
class WrongDivisor(Solution):
    def flip(self):
        self.total -= 1
        x = random.randint(0, self.total)
        idx = self.mp.get(x, x)
        self.mp[x] = self.mp.get(self.total, self.total)
        return [idx // self.m, idx % self.m]           # SAI: chia cho m thay vì n


random.seed(1)
for m, n in [(3, 3), (2, 5)]:
    s = WrongDivisor(m, n)
    got = [tuple(s.flip()) for _ in range(m * n)]
    ok = len(set(got)) == m * n and all(0 <= i < m and 0 <= j < n for i, j in got)
    print(f"lưới {m}x{n}: hợp lệ? {ok}", "" if ok else f"-> ví dụ ô lỗi: {[c for c in got if not (0 <= c[0] < m and 0 <= c[1] < n)][:2]}")

lưới 3x3: hợp lệ? True 
lưới 2x5: hợp lệ? False -> ví dụ ô lỗi: [(3, 1), (3, 0)]


**Bài học:** luôn test lưới **không vuông** (như `2 x 5`, `5 x 2`) — mục 12 đã làm.

### Lỗi 5: "bốc bừa đến khi trúng" (Cách 1/3) rồi tưởng đã tối ưu

Nó đúng về xác suất nhưng **số lần random không bị chặn**. Với lưới `1 x 1000` và đã lật 999 ô, lần cuối phải thử trung bình `1000` lần — đề yêu cầu tối thiểu số lần gọi random, nên đây là cách bị đánh giá kém (mục 11).

### Lỗi 6: test ngẫu nhiên bằng so sánh đáp án cứng

`assert sol.flip() == [1, 0]` sẽ đúng hôm nay (khi may mắn) rồi sai ngày mai. Hãy test **tính chất** (trong lưới, chưa lật, không trùng) và **thống kê/vét cạn** (mục 12).

## 14. Mẹo & nhận dạng pattern

- **"Bốc ngẫu nhiên đều, không bốc lại, `N` rất lớn mà chỉ bốc ít lần"** ⇒ **Fisher–Yates ảo** với dict (còn gọi *sparse Fisher–Yates*, *lazy shuffle*).
- **2D → 1D:** gặp lưới + chọn ngẫu nhiên, hãy phẳng hoá thành `[0, m·n)`; `idx = i*n + j`, `divmod(idx, n)`. Giúp giảm số lần random từ 2 xuống 1 và đơn giản hoá việc loại trùng.
- **"Mảng ảo" bằng dict + `get(k, k)`:** mẹo chung cho mọi bài cần mảng khổng lồ nhưng chỉ vài phần tử bị sửa.
- **"Tối thiểu số lần gọi random"** là tín hiệu: bốc lại (rejection sampling) **không** được chấp nhận; phải ánh xạ đúng một giá trị random thành đúng một kết quả hợp lệ.
- **Khi `randint` gồm cả hai đầu,** luôn tự hỏi "vùng hợp lệ có bao nhiêu phần tử?" và viết `randint(0, size-1)` (hoặc giảm `size` trước).
- **Test ngẫu nhiên:** (1) `seed` cố định; (2) tính chất bất biến; (3) vét cạn bằng RNG kịch bản khi không gian nhỏ; (4) thống kê ≤ 5σ.
- Cùng họ: bài 384 (Shuffle an Array), 380 (Insert Delete GetRandom O(1)), 710 (Random Pick with Blacklist), 398, 382.

## 15. Biến thể

1. **Bài 710 — Random Pick with Blacklist:** bốc đều một số trong `[0, N)` **không thuộc** blacklist. Cùng dạng "mảng ảo", ánh xạ các số bị cấm sang các số hợp lệ ở cuối.
2. **Có hoàn lại (with replacement):** `flip` không đánh dấu ô — chỉ cần `divmod(random.randint(0, N-1), n)`; không cần dict.
3. **Lấy `k` ô khác nhau một lúc** (`sample`): lặp `flip` `k` lần — chính là Fisher–Yates cắt ngắn, `O(k)`.
4. **Hỗ trợ thêm `unflip(i, j)`** (đưa một ô về `0`): khó hơn nhiều — cần cấu trúc chỉ mục hai chiều (dict ngược `vị trí → giá trị`), xem như bài tập mở.
5. **Lưới nhiều chiều:** `(i, j, k)` ↔ `idx = (i*n + j)*p + k` — cùng thuật toán.
6. **Trọng số khác nhau:** bài 528 (Random Pick with Weight) dùng prefix sum + tìm nhị phân.

Minh hoạ nhanh biến thể 2 (có hoàn lại) và 3 (lấy `k` ô khác nhau) bằng cùng một ý tưởng:

In [28]:
def pick_with_replacement(m, n, rng):
    return list(divmod(rng.randint(0, m * n - 1), n))


def sample_cells(m, n, k, rng):
    """Biến thể 3: k ô khác nhau, đều — Fisher-Yates ảo dừng sau k bước."""
    total, mp, out = m * n, {}, []
    for _ in range(k):
        total -= 1
        x = rng.randint(0, total)
        idx = mp.get(x, x)
        mp[x] = mp.get(total, total)
        out.append(divmod(idx, n))
    return out


g = LCG(3)
print([pick_with_replacement(2, 3, g) for _ in range(5)])      # có thể trùng
cells = sample_cells(3, 3, 5, g)
print(cells)
assert len(set(cells)) == 5

[[0, 0], [0, 1], [1, 1], [0, 2], [1, 0]]
[(1, 0), (1, 1), (2, 2), (2, 0), (1, 2)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time `flip` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Lưới + bốc bừa | giữ lưới thật, thử đến khi trúng `0` | kỳ vọng `N/(N-k)` thử | `O(N)` | chỉ để học/lưới rất nhỏ |
| 2. List ô trống | Fisher–Yates thật trên list `N` phần tử | `O(1)` | `O(N)` | `N` nhỏ (≤ vài triệu), cần đơn giản và đúng 1 random |
| 3. Set + bốc lại | nhớ ô đã lật, bốc lại nếu trùng | kỳ vọng `N/(N-k)` | `O(k)` | `k ≪ N` luôn đúng; không cần chặt số lần random |
| 4. Hạng thứ r | bốc hạng, dịch qua danh sách đã lật (sắp xếp) | `O(k)` | `O(k)` | `k` nhỏ, muốn 1 random mà không dùng dict |
| 5. Fisher–Yates ảo | mảng ảo `A[k]=k`, dict lưu các ô bị ghi đè | `O(1)` | `O(k)` | **tổng quát, tối ưu — dùng khi nộp bài** |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được chạy, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Với `Solution` ở mục 9 và lưới `m = 1`, `n = 4`, kịch bản bốc `x = 2, 0, 1, 0` (ép bằng `scripted_random`), `flip` trả về lần lượt gì? Hãy tự dự đoán trước rồi chạy dòng kiểm tra ở mục 19.

In [29]:
# TODO Bài 1: ghi dự đoán của bạn (list gồm 4 toạ độ [i, j]) vào biến này
bai1_du_doan = None

### Bài 2 (tính tay)

Một lưới có `N = 5` ô; ta lật **hết** cả 5 ô bằng Cách 3 (bốc cả khoảng rồi bốc lại). Số lần thử kỳ vọng bằng `N·(1 + 1/2 + ... + 1/N)`. Hãy tính tay giá trị này (làm tròn 2 chữ số thập phân) rồi viết hàm `ky_vong(N)` trả đúng giá trị đó.

In [30]:
# TODO Bài 2: viết hàm ky_vong(N)
def ky_vong(N):
    pass

### Bài 3 (điền TODO)

Hoàn thành `flip` của lớp Fisher–Yates ảo (đừng quên cập nhật `mp`).

In [31]:
class FlipTODO:
    def __init__(self, m, n, rng):
        self.m, self.n, self.rng = m, n, rng
        self.total = m * n
        self.mp = {}

    def flip(self):
        # TODO: giảm total, bốc x, lấy idx = giá trị tại vị trí x, lấp chỗ trống, trả [hàng, cột]
        pass

    def reset(self):
        # TODO
        pass

### Bài 4 (tự viết lại)

Không nhìn mục 9, hãy tự viết lớp `MyFlip(m, n)` dùng `random.randint` (cùng thuật toán, 1 lần random mỗi `flip`, `reset` đúng). Cell dưới chỉ để trống khung.

In [32]:
# TODO Bài 4: tự viết lại lớp
class MyFlip:
    pass

### Bài 5 (biến thể / mini challenge)

Viết hàm `sample_without_replacement(N, k, rng)` trả về **`k` số khác nhau** đều trong `[0, N)` (với `N` có thể tới `10^9`, `k ≤ 1000`), dùng đúng `k` lần gọi `rng.randint` và bộ nhớ `O(k)`. Sau đó tự trả lời: nếu cần **`flip` rồi `unflip`** thì điều gì làm khó cấu trúc `mp`?

In [33]:
# TODO Bài 5: viết hàm
def sample_without_replacement(N, k, rng):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `total` bắt đầu bằng `4`; mỗi lần `flip` giảm `total` trước rồi mới bốc.
- Mức 2: theo dõi `mp` như bảng ở mục 8: `idx = mp.get(x, x)`; `mp[x] = mp.get(total, total)`.
- Mức 3: lần 1: `total=3`, `x=2` ⇒ `idx=2`, `mp={2:3}`; lần 2: `total=2`, `x=0` ⇒ `idx=0`, `mp[0]=mp.get(2,2)=3` nên `mp={2:3, 0:3}`; lần 3: `total=1`, `x=1` ⇒ `idx=1`; lần 4: `total=0`, `x=0` ⇒ `idx=mp[0]=3`. Vì `n = 4` và `m = 1` nên toạ độ là `[0, idx]`.

**Bài 2**
- Mức 1: cộng `1 + 1/2 + 1/3 + 1/4 + 1/5` (số điều hoà).
- Mức 2: tổng này ≈ `2.2833`.
- Mức 3: nhân với `N = 5` ⇒ ≈ `11.42`; code: `N * sum(1/k for k in range(1, N+1))`.

**Bài 3**
- Mức 1: giống code nộp nhưng thay `random.randint` bằng `self.rng.randint`.
- Mức 2: ba dòng cốt lõi: `self.total -= 1`; `x = ...`; `idx = self.mp.get(x, x)`.
- Mức 3: dòng thứ tư `self.mp[x] = self.mp.get(self.total, self.total)`; trả `[idx // self.n, idx % self.n]`; `reset` đặt `total` và xoá `mp`.

**Bài 4**
- Mức 1: cần 3 thuộc tính: kích thước, `total`, dict `mp`.
- Mức 2: `flip` có đúng 5 dòng; `reset` có 2 dòng.
- Mức 3: xem lại mục 6 (bốn bước: giảm `total`, bốc `x`, lấy `A[x]`, lấp chỗ trống).

**Bài 5**
- Mức 1: đây là `flip` lặp `k` lần trên "lưới" `1 x N`.
- Mức 2: dùng `total = N`, `mp = {}` và vòng lặp `for _ in range(k)`.
- Mức 3: kết quả mỗi bước chính là `idx`; trả về danh sách `idx`. Với `unflip`: phần tử đã lật không còn nằm trong "vùng hợp lệ"; để trả nó lại phải biết **vị trí hiện tại** của nó (ánh xạ ngược), tức phải duy trì thêm dict ngược `giá trị → vị trí`.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [34]:
# Đáp án Bài 1: lưới 1x4, picks 2, 0, 1, 0
with scripted_random([2, 0, 1, 0]):
    s = Solution(1, 4)
    bai1_dap_an = [s.flip() for _ in range(4)]
print(bai1_dap_an)                       # [[0, 2], [0, 0], [0, 1], [0, 3]]
assert bai1_dap_an == [[0, 2], [0, 0], [0, 1], [0, 3]]
assert sorted(c[1] for c in bai1_dap_an) == [0, 1, 2, 3]     # phủ đủ 4 ô

[[0, 2], [0, 0], [0, 1], [0, 3]]


In [35]:
# Đáp án Bài 2
def ky_vong(N):
    return round(N * sum(1 / k for k in range(1, N + 1)), 2)


print(ky_vong(5))
assert ky_vong(5) == 11.42 and ky_vong(1) == 1.0 and ky_vong(9) == round(expected_attempts_all(9), 2)

11.42


In [36]:
# Đáp án Bài 3
class FlipTODO:
    def __init__(self, m, n, rng):
        self.m, self.n, self.rng = m, n, rng
        self.total = m * n
        self.mp = {}

    def flip(self):
        self.total -= 1
        x = self.rng.randint(0, self.total)
        idx = self.mp.get(x, x)
        self.mp[x] = self.mp.get(self.total, self.total)
        return [idx // self.n, idx % self.n]

    def reset(self):
        self.total = self.m * self.n
        self.mp.clear()


check_full_cycle(lambda m, n: FlipTODO(m, n, LCG(9)), 3, 4)
assert [FlipTODO(1, 3, ScriptedRNG([1, 1, 0])).flip()] == [[0, 1]]
print("Bài 3: OK")

# Đáp án Bài 4
class MyFlip:
    def __init__(self, m, n):
        self.m, self.n = m, n
        self.total = m * n
        self.mp = {}

    def flip(self):
        self.total -= 1
        x = random.randint(0, self.total)
        idx = self.mp.get(x, x)
        self.mp[x] = self.mp.get(self.total, self.total)
        return [idx // self.n, idx % self.n]

    def reset(self):
        self.total = self.m * self.n
        self.mp.clear()


random.seed(4)
for m, n in [(1, 1), (2, 5), (5, 2), (4, 4)]:
    check_full_cycle(MyFlip, m, n)
print("Bài 4: OK")

Bài 3: OK
Bài 4: OK


In [37]:
# Đáp án Bài 5
def sample_without_replacement(N, k, rng):
    total, mp, out = N, {}, []
    for _ in range(k):
        total -= 1
        x = rng.randint(0, total)
        idx = mp.get(x, x)
        mp[x] = mp.get(total, total)
        out.append(idx)
    return out


g = LCG(2026)
res = sample_without_replacement(10 ** 9, 1000, g)
assert len(set(res)) == 1000 and all(0 <= v < 10 ** 9 for v in res) and g.calls == 1000
full = sample_without_replacement(6, 6, LCG(1))
assert sorted(full) == [0, 1, 2, 3, 4, 5]

# đều: bốc 2 phần tử từ N = 4 -> 12 cặp có thứ tự, mỗi cặp ~ trials/12
trials = 48000
g = LCG(31)
cnt = Counter(tuple(sample_without_replacement(4, 2, g)) for _ in range(trials))
keys = [(a, b) for a in range(4) for b in range(4) if a != b]
assert worst_sigma(cnt, keys, trials) < 5
print("Bài 5: OK")

Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "Fisher–Yates ảo" (bốc không hoàn lại từ `[0, N)` rất lớn):**

```python
import random

class Solution:
    def __init__(self, m: int, n: int):
        self.m, self.n = m, n
        self.total = m * n          # số phần tử còn lại
        self.mp = {}                # mảng ảo: vị trí -> giá trị (vắng mặt = chính nó)

    def flip(self) -> List[int]:
        self.total -= 1                              # 1) thu nhỏ vùng còn lại: A[0..total]
        x = random.randint(0, self.total)            # 2) bốc vị trí x (gồm cả total)
        idx = self.mp.get(x, x)                      # 3) giá trị tại x
        self.mp[x] = self.mp.get(self.total, self.total)   # 4) lấp chỗ trống bằng phần tử cuối
        return [idx // self.n, idx % self.n]         # 5) chia cho SỐ CỘT n

    def reset(self) -> None:
        self.total = self.m * self.n
        self.mp.clear()
```

**Checklist trước khi submit:**

- [ ] Đã phẳng hoá lưới `idx = i*n + j` và chia lại bằng **`n`** (số cột), không phải `m`?
- [ ] `total` giảm **trước** khi bốc và `randint(0, total)` (gồm cả hai đầu) — không bốc ra ngoài vùng còn lại?
- [ ] Có dòng "lấp chỗ trống" `mp[x] = mp.get(total, total)`?
- [ ] `reset` đặt lại **cả** `total` **và** xoá `mp`?
- [ ] Mỗi `flip` chỉ gọi random **đúng một lần**; không có vòng lặp "bốc lại"?
- [ ] Bộ nhớ chỉ phụ thuộc số lần `flip` (không tạo list/ma trận cỡ `m·n`)?
- [ ] Đã test: `1 x 1`, lưới không vuông (`2 x 5`), lật hết rồi `reset`, vét cạn/thống kê với seed?
- [ ] Đã `import random` ở đầu file (nếu môi trường chưa nạp sẵn)?